# VigiLense: Fire & Smoke detection (D-Fire) with YOLO26

**Files needed in Google Drive, folder `My Drive/vigilense/datasets/`:**
1. `D-Fire-images.zip`: the big download called **"D-Fire dataset (only images and labels)"** from the D-Fire GitHub page.
2. `dfire_train.txt` and `dfire_test.txt`: the official split lists, from the small download called "Training, validation and test sets". (Ignore `video_*.txt` and the `5-fold` folder.)

Then in Colab: **Runtime > Change runtime type > T4 GPU**.

Run the cells top to bottom. Start with `QUICK_TEST = True` (a few minutes) to prove everything works, then set it to `False` for the real run.

Cite the D-Fire paper in your report (Venancio et al., Neural Computing and Applications, 2022).

In [ ]:
# 1) GPU check + install + connect Google Drive
!nvidia-smi
!pip install -U -q ultralytics

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Paths and settings
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/vigilense"
DS         = f"{DRIVE_ROOT}/datasets"
ZIP_PATH   = f"{DS}/D-Fire-images.zip"
RAW_DIR    = Path("/content/D-Fire")        # extracted dataset (local disk = fast)
CFG_DIR    = Path("/content/dfire_cfg")     # split lists + data.yaml

QUICK_TEST = False    # True = 3-epoch test on 10% of data. Set False for real training.
EPOCHS     = 3 if QUICK_TEST else 50
RUN_NAME   = "fire_smoke_quick" if QUICK_TEST else "fire_smoke_v1"

for f in ["D-Fire-images.zip", "dfire_train1.txt", "dfire_valid1.txt", "dfire_test.txt"]:
    assert Path(f"{DS}/{f}").exists(), f"Missing in Drive: {f}"
print("All files found. QUICK_TEST =", QUICK_TEST)

In [ ]:
# 3) Copy the zip to local disk and unzip
import os, shutil, subprocess

LOCAL_ZIP = "/content/D-Fire-images.zip"

if (RAW_DIR / "train" / "images").exists() and (RAW_DIR / "test" / "images").exists():
    print("Dataset already extracted, skipping.")
else:
    if not os.path.exists(LOCAL_ZIP) or os.path.getsize(LOCAL_ZIP) < 2_500_000_000:
        print("Copying zip from Drive...")
        shutil.copy(ZIP_PATH, LOCAL_ZIP)
    print("Unzipping...")
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(["unzip", "-o", "-q", LOCAL_ZIP, "-d", str(RAW_DIR)], capture_output=True, text=True)
    print(r.stderr[-500:])

n_files = sum(1 for p in RAW_DIR.rglob("*") if p.is_file())
print("Files in dataset:", n_files)       # expect ~43,054
assert n_files > 40000, "Extraction looks incomplete."

In [ ]:
# 4) Build train / val / test folders from the official split lists
import yaml

CFG_DIR.mkdir(parents=True, exist_ok=True)

def build(list_file, subdir, out_name):
    paths, missing = [], 0
    for line in Path(f"{DS}/{list_file}").read_text().splitlines():
        name = Path(line.strip().replace("\\", "/")).name
        if not name:
            continue
        p = RAW_DIR / subdir / "images" / name
        if p.exists():
            paths.append(str(p))
        else:
            missing += 1
    (CFG_DIR / out_name).write_text("\n".join(paths))
    print(f"{out_name:10s}: {len(paths)} images | missing: {missing}")
    return paths

train_paths = build("dfire_train1.txt", "train", "train.txt")
val_paths   = build("dfire_valid1.txt", "train", "val.txt")
test_paths  = build("dfire_test.txt",   "test",  "test.txt")
print("train/val overlap (must be 0):", len(set(train_paths) & set(val_paths)))

DATA_YAML = CFG_DIR / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "train": str(CFG_DIR / "train.txt"),
    "val":   str(CFG_DIR / "val.txt"),
    "test":  str(CFG_DIR / "test.txt"),
    "names": {0: "smoke", 1: "fire"},
}, sort_keys=False))
print(DATA_YAML.read_text())

In [ ]:
# 5) Sanity check: counts, class ids, and empty labels (= negative images)
from collections import Counter

def label_stats(paths):
    classes, empty = Counter(), 0
    for p in paths:
        lp = Path(p.replace("/images/", "/labels/")).with_suffix(".txt")
        lines = [l for l in lp.read_text().splitlines() if l.strip()] if lp.exists() else []
        if not lines:
            empty += 1
        classes.update(l.split()[0] for l in lines)
    return dict(classes), empty

for name, ps in [("train", train_paths), ("val", val_paths), ("test", test_paths)]:
    cls, empty = label_stats(ps)
    print(f"{name:5s}: {len(ps)} images | boxes per class {cls} | background images: {empty}")

print("\nExpected: only class ids '0' (smoke) and '1' (fire).")

In [ ]:
# 6) Train (results and checkpoints are saved straight to Google Drive)
from ultralytics import YOLO

model = YOLO("yolo26n.pt")      # if this fails to download, use "yolo11n.pt"
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    patience=15,
    fraction=0.1 if QUICK_TEST else 1.0,
    project=f"{DRIVE_ROOT}/runs",
    name=RUN_NAME,
    exist_ok=True,
    workers=2,
    plots=True,
)

In [ ]:
# 7) Evaluate the best model on the TEST split
BEST = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt"
metrics = YOLO(BEST).val(data=str(DATA_YAML), split="test", imgsz=640)
print(f"mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f} | Recall: {metrics.box.mr:.3f}")

In [ ]:
# 8) Look at predictions on random test images
import random
from PIL import Image
from IPython.display import display

best_model = YOLO(BEST)
for p in random.sample(test_paths, 6):
    r = best_model.predict(p, conf=0.25, verbose=False)[0]
    display(Image.fromarray(r.plot()[..., ::-1]).resize((640, 480)))

## If Colab disconnects during a long run
Re-run cells 1 and 2 (and 3 to 4 if the local dataset is gone), then run the cell below to continue from the last checkpoint. Set `QUICK_TEST = False` first.

In [ ]:
# Resume an interrupted training run
last = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/last.pt"
YOLO(last).train(resume=True)

## After training
`best.pt` is in Google Drive at `vigilense/runs/<run name>/weights/best.pt`. Download it, put it in your project's `weights/` folder, and test on your laptop:

`yolo predict model=weights\best.pt source="data\test.mp4" show=True`